# BPA Nap3 – Zoho és CMC
Előkészített, még nem futtatott Colab-jegyzetfüzet. A helyi tesztek nem Colab-bizonyítékok.
Csak fiktív tanfolyami rekordokat hoz létre. Nem küld üzenetet és nincs időzítő.
Futtasd a cellákat sorrendben; **ne használd a Run all parancsot** a beviteli cellák miatt.
Az API Teszt, a mini-feladat és a két árfolyamág külön futás.
Titkot kizárólag a saját maszkolt bemeneti meződbe írj. `.env.local` feltöltése tilos.
A Colab-runtime újraindítása törli a memóriabeli hozzáférést. Mentés/megosztás előtt töröld a kimeneteket.


In [ ]:
print("Szia, BPA!")


A következő cella a megvizsgált közös kliens és az automatizmusok pontos forrását tölti be. Nem olvas helyi környezeti fájlt, nem küld hálózati kérést.


In [ ]:
import sys, types
MODULE_SOURCES = {'bpa_access': '"""Shared course API access. Standard library only; never prints credentials.\n\nCommands from the BPA root:\n  python nap03/working/bpa_access.py status\n  python nap03/working/bpa_access.py check\n  python nap03/working/bpa_access.py zoho-auth\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport re\nimport subprocess\nimport time\nfrom pathlib import Path\nfrom urllib.error import HTTPError, URLError\nfrom urllib.parse import urlencode\nfrom urllib.request import HTTPRedirectHandler, Request, build_opener\n\nROOT = Path(__file__).resolve().parents[2]\nENV_FILE = ROOT / \'.env.local\'\nTOKEN_HELPER = Path(__file__).with_name(\'zoho_tokens.ps1\')\nKEYS = (\'CMC_PRO_API_KEY\', \'ZOHO_CLIENT_ID\', \'ZOHO_CLIENT_SECRET\',\n        \'ZOHO_GRANT_CODE\', \'ZOHO_ACCESS_TOKEN\', \'ZOHO_REFRESH_TOKEN\')\n\n\nclass AccessError(RuntimeError):\n    """Only safe, predefined diagnostics may be included in this exception."""\n\n\nclass APIError(AccessError):\n    def __init__(self, service: str, status: int, code: str = \'\'):\n        self.status = status\n        self.code = code\n        # Never echo a provider message, URL, response body, or arbitrary code.\n        safe_codes = {\'INVALID_TOKEN\', \'OAUTH_SCOPE_MISMATCH\', \'INVALID_MODULE\',\n                      \'NO_PERMISSION\', \'AUTHENTICATION_FAILURE\', \'INVALID_DATA\',\n                      \'DUPLICATE_DATA\', \'MANDATORY_NOT_FOUND\'}\n        suffix = f\' ({code})\' if code in safe_codes else \'\'\n        super().__init__(f\'{service} request failed: HTTP {status}{suffix}.\')\n\n\ndef read_env(path: Path = ENV_FILE) -> dict[str, str]:\n    if not path.is_file():\n        raise AccessError(\'Root .env.local is missing.\')\n    values: dict[str, str] = {}\n    for line in path.read_text(encoding=\'utf-8-sig\').splitlines():\n        match = re.fullmatch(r\'\\s*([A-Z][A-Z0-9_]*)\\s*=(.*)\', line)\n        if not match:\n            if line.strip() and not line.lstrip().startswith(\'#\'):\n                raise AccessError(\'Unsupported .env.local syntax; use KEY=value.\')\n            continue\n        key, value = match.groups()\n        if key in values:\n            raise AccessError(f\'Duplicate field in .env.local: {key}.\')\n        value = value.strip()\n        if len(value) >= 2 and value[0] == value[-1] and value[0] in "\\"\'":\n            value = value[1:-1]\n        values[key] = value\n    return values\n\n\nclass NoRedirect(HTTPRedirectHandler):\n    def redirect_request(self, req, fp, code, msg, headers, newurl):\n        # Prevent authorization headers being forwarded to another host.\n        return None\n\n\nclass CourseAPI:\n    def __init__(self, env_file: Path = ENV_FILE):\n        self.env_file = Path(env_file)\n        self.opener = build_opener(NoRedirect())\n        self.last_status: int | None = None\n\n    def credentials(self) -> dict[str, str]:\n        return read_env(self.env_file)\n\n    def authenticate(self, *, force: bool = False) -> str:\n        values = self.credentials()\n        token = values.get(\'ZOHO_ACCESS_TOKEN\', \'\')\n        try:\n            expires_at = int(values.get(\'ZOHO_ACCESS_TOKEN_EXPIRES_AT\', \'0\'))\n        except ValueError:\n            expires_at = 0\n        if token and not force and (not expires_at or expires_at > time.time() + 120):\n            return token\n        for key in (\'ZOHO_CLIENT_ID\', \'ZOHO_CLIENT_SECRET\'):\n            if not values.get(key):\n                raise AccessError(f\'Fill {key} in .env.local.\')\n        if not values.get(\'ZOHO_REFRESH_TOKEN\') and not values.get(\'ZOHO_GRANT_CODE\'):\n            raise AccessError(\'Zoho needs initial authorization: generate a fresh EU Self Client code, save it as ZOHO_GRANT_CODE, then rerun the access check.\')\n        try:\n            result = subprocess.run(\n                [\'powershell.exe\', \'-NoProfile\', \'-NonInteractive\', \'-File\',\n                 str(TOKEN_HELPER), \'-Action\', \'Auto\', \'-EnvFile\', str(self.env_file)],\n                capture_output=True, timeout=45, check=False,\n            )\n        except (OSError, subprocess.TimeoutExpired):\n            raise AccessError(\'Zoho token helper could not run or timed out. Check PowerShell and connectivity.\') from None\n        if result.returncode:\n            # Child output is intentionally suppressed: no credentials or tracebacks.\n            raise AccessError(\'Zoho authorization failed. Check EU Client ID/Secret and use a fresh unused grant code, or check whether the refresh token was revoked.\')\n        values = self.credentials()\n        if not values.get(\'ZOHO_ACCESS_TOKEN\'):\n            raise AccessError(\'Zoho token helper did not save an access token.\')\n        return values[\'ZOHO_ACCESS_TOKEN\']\n\n    def _request(self, service: str, method: str, url: str,\n                 headers: dict[str, str], payload: dict | None = None) -> dict:\n        data = None if payload is None else json.dumps(payload).encode(\'utf-8\')\n        request = Request(url, data=data, headers=headers, method=method)\n        self.last_status = None\n        try:\n            with self.opener.open(request, timeout=30) as response:\n                self.last_status = response.status\n                raw = response.read()\n        except HTTPError as error:\n            self.last_status = error.code\n            code = \'\'\n            try:\n                code = json.loads(error.read()).get(\'code\', \'\')\n            except (ValueError, AttributeError):\n                pass\n            raise APIError(service, error.code, code) from None\n        except (URLError, OSError, TimeoutError):\n            note = \' Check the service before retrying a write.\' if method != \'GET\' else \'\'\n            raise AccessError(f\'{service} network request failed.{note}\') from None\n        if not raw:\n            return {}\n        try:\n            result = json.loads(raw)\n        except ValueError:\n            raise AccessError(f\'{service} returned a non-JSON response.\') from None\n        if not isinstance(result, dict):\n            raise AccessError(f\'{service} returned an unexpected response shape.\')\n        return result\n\n    def cmc_quotes(self, ids: tuple[int, ...] = (1, 1027), convert: str = \'EUR\') -> dict:\n        if not ids or any(type(item) is not int or item < 1 for item in ids):\n            raise AccessError(\'CoinMarketCap IDs must be positive integers.\')\n        if not re.fullmatch(r\'[A-Z]{3}\', convert):\n            raise AccessError(\'Use a three-letter uppercase currency code.\')\n        key = self.credentials().get(\'CMC_PRO_API_KEY\')\n        if not key:\n            raise AccessError(\'Fill CMC_PRO_API_KEY in .env.local.\')\n        url = \'https://pro-api.coinmarketcap.com/v1/cryptocurrency/quotes/latest?\' + urlencode(\n            {\'id\': \',\'.join(map(str, ids)), \'convert\': convert})\n        result = self._request(\'CoinMarketCap\', \'GET\', url,\n                               {\'X-CMC_PRO_API_KEY\': key, \'Accept\': \'application/json\'})\n        if result.get(\'status\', {}).get(\'error_code\') != 0:\n            raise AccessError(\'CoinMarketCap rejected the query. Check the key, plan, and quota.\')\n        for coin_id in ids:\n            try:\n                float(result[\'data\'][str(coin_id)][\'quote\'][convert][\'price\'])\n            except (KeyError, TypeError, ValueError):\n                raise AccessError(\'CoinMarketCap response is missing a requested price.\') from None\n        return result\n\n    def zoho(self, method: str, resource: str, *, params: dict | None = None,\n             payload: dict | None = None) -> dict:\n        method = method.upper()\n        if method not in {\'GET\', \'POST\', \'PUT\', \'DELETE\'}:\n            raise AccessError(\'Unsupported CRM request method.\')\n        if not re.fullmatch(r\'[A-Za-z][A-Za-z0-9_]*(?:/[A-Za-z0-9_]+)*\', resource):\n            raise AccessError(\'Use a CRM resource such as Leads or Leads/record_id.\')\n        token = self.authenticate()\n        url = \'https://www.zohoapis.eu/crm/v8/\' + resource\n        if params:\n            url += \'?\' + urlencode(params)\n        headers = {\'Authorization\': \'Zoho-oauthtoken \' + token,\n                   \'Accept\': \'application/json\', \'Content-Type\': \'application/json\'}\n        try:\n            return self._request(\'Zoho\', method, url, headers, payload)\n        except APIError as error:\n            if method == \'GET\' and error.status == 401 and error.code == \'INVALID_TOKEN\':\n                headers[\'Authorization\'] = \'Zoho-oauthtoken \' + self.authenticate(force=True)\n                return self._request(\'Zoho\', method, url, headers, payload)\n            # Never automatically retry creates/updates after uncertain outcomes.\n            raise\n\n\ndef check_access(api: CourseAPI) -> dict:\n    report = {}\n    try:\n        api.cmc_quotes()\n        report[\'coinmarketcap\'] = {\'status\': \'verified\', \'check\': \'BTC and ETH quotes in EUR\'}\n    except AccessError as error:\n        report[\'coinmarketcap\'] = {\'status\': \'needs_attention\', \'detail\': str(error)}\n    try:\n        # Request just one field; never print or persist returned CRM records.\n        api.zoho(\'GET\', \'Leads\', params={\'fields\': \'Last_Name\', \'per_page\': 1})\n        report[\'zoho_crm\'] = {\'status\': \'verified\', \'check\': \'EU CRM Leads read access\'}\n        if not api.credentials().get(\'ZOHO_REFRESH_TOKEN\'):\n            report[\'zoho_crm\'][\'refresh\'] = \'missing: unattended token renewal is not ready\'\n    except AccessError as error:\n        report[\'zoho_crm\'] = {\'status\': \'needs_attention\', \'detail\': str(error)}\n    return report\n\n\ndef main() -> int:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'command\', choices=(\'status\', \'check\', \'zoho-auth\'))\n    args = parser.parse_args()\n    api = CourseAPI()\n    try:\n        if args.command == \'status\':\n            values = api.credentials()\n            print(json.dumps({key: \'filled\' if values.get(key) else \'empty\' for key in KEYS}, indent=2))\n        elif args.command == \'zoho-auth\':\n            api.authenticate(force=True)\n            print(\'Zoho token saved locally; credential values were not displayed.\')\n        else:\n            report = check_access(api)\n            print(json.dumps(report, indent=2))\n            return 0 if all(item[\'status\'] == \'verified\' for item in report.values()) else 1\n    except AccessError as error:\n        print(str(error))\n        return 1\n    return 0\n\n\nif __name__ == \'__main__\':\n    raise SystemExit(main())\n', 'api_exercises': '"""Course exercise logic shared by local checks and the actual Colab notebook.\n\nOnly fictional .example identities are accepted. Never retries a write. A failed\nwrite reserves its identity in memory until a person reconciles the CRM result.\n"""\nfrom datetime import datetime, timezone\nimport math\nimport re\n\nfrom bpa_access import AccessError\n\nSAMPLE = {\n    \'Last_Name\': \'Teszt\', \'First_Name\': \'API\', \'Company\': \'Atlas Market\',\n    \'Email\': \'api.teszt@example.com\', \'Lead_Source\': \'API\',\n    \'Description\': \'Oktatási rekord, a harmadik napon Colabból hoztam létre.\',\n}\nMINI = {\n    \'Last_Name\': \'Kovács\', \'First_Name\': \'BPA\', \'Company\': \'Duna Minta Kft\',\n    \'Email\': \'bpa.kovacs@dunaminta.example\', \'Lead_Source\': \'API\',\n    \'Description\': \'Fiktív üzleti példa: érdeklődés egy automatizált készletjelentés iránt.\',\n}\n\n\ndef now():\n    return datetime.now(timezone.utc).isoformat()\n\n\ndef validate_lead(lead):\n    required = {\'Last_Name\', \'Company\', \'Email\', \'Lead_Source\', \'Description\'}\n    if not required <= lead.keys() or any(not lead[key] for key in required):\n        raise AccessError(\'Missing required fictional course lead fields.\')\n    if not re.fullmatch(r\'[a-z0-9.]+@(?:[a-z0-9]+\\.example|example\\.com)\', lead[\'Email\']):\n        raise AccessError(\'Use a reserved fictional example email identity.\')\n    if lead[\'Lead_Source\'] != \'API\':\n        raise AccessError(\'These API exercises require API as Lead Source.\')\n\n\ndef find_exact(api, email):\n    # Search may return punctuation variants: independently compare the identity.\n    result = api.zoho(\'GET\', \'Leads/search\', params={\n        \'email\': email, \'fields\': \'id,First_Name,Last_Name,Company,Email,Lead_Source,Description\',\n        \'per_page\': 200,\n    })\n    if result.get(\'info\', {}).get(\'more_records\'):\n        raise AccessError(\'Identity search was truncated; reconcile before writing.\')\n    return [row for row in result.get(\'data\', []) if row.get(\'Email\', \'\').lower() == email.lower()]\n\n\ndef check_fields(row, lead):\n    if any(row.get(key) != value for key, value in lead.items()):\n        raise AccessError(\'Existing/readback course lead fields differ; inspect CRM before retrying.\')\n\n\ndef create_verified(api, lead, attempts):\n    validate_lead(lead)\n    identity = lead[\'Email\']\n    existing = find_exact(api, identity)\n    if len(existing) > 1:\n        raise AccessError(\'Multiple records share the fictional identity; reconcile before writing.\')\n    if existing:\n        check_fields(existing[0], lead)\n        return {\'outcome\': \'existing_exact_record\', \'created_this_run\': False,\n                \'record_id\': existing[0][\'id\'], \'fields\': dict(lead), \'checked_at\': now()}\n    if identity in attempts:\n        raise AccessError(\'A write was already attempted for this identity; reconcile CRM before retrying.\')\n    attempts[identity] = {\'state\': \'attempted_requires_reconciliation\', \'at\': now()}\n    result = api.zoho(\'POST\', \'Leads\', payload={\'data\': [lead], \'trigger\': []})\n    http_status = api.last_status\n    rows = result.get(\'data\', [])\n    if http_status != 201 or len(rows) != 1:\n        raise AccessError(\'Expected HTTP201 and one CRM result; reconcile before retrying.\')\n    item = rows[0]\n    record_id = item.get(\'details\', {}).get(\'id\', \'\')\n    if item.get(\'status\') != \'success\' or item.get(\'code\') != \'SUCCESS\' or not re.fullmatch(r\'\\d+\', record_id):\n        raise AccessError(\'CRM did not confirm one successful record ID; reconcile before retrying.\')\n    attempts[identity].update(state=\'created_pending_readback\', record_id=record_id)\n    readback = api.zoho(\'GET\', \'Leads/\' + record_id).get(\'data\', [])\n    if len(readback) != 1 or readback[0].get(\'id\') != record_id:\n        raise AccessError(\'Exact CRM readback missing; do not resubmit.\')\n    check_fields(readback[0], lead)\n    attempts[identity][\'state\'] = \'verified\'\n    return {\'outcome\': \'created_and_verified\', \'created_this_run\': True,\n            \'http_status\': http_status, \'record_id\': record_id, \'fields\': dict(lead), \'checked_at\': now()}\n\n\ndef finite_positive(value):\n    if isinstance(value, bool):\n        raise AccessError(\'Price/threshold must be a finite positive number.\')\n    try:\n        number = float(value)\n    except (ValueError, TypeError):\n        raise AccessError(\'Price/threshold must be a finite positive number.\') from None\n    if not math.isfinite(number) or number <= 0:\n        raise AccessError(\'Price/threshold must be a finite positive number.\')\n    return number\n\n\ndef above_threshold(price, threshold):\n    return finite_positive(price) > finite_positive(threshold)\n\n\ndef fetch_btc(api):\n    result = api.cmc_quotes((1,), \'EUR\')\n    return {\'price_eur\': finite_positive(result[\'data\'][\'1\'][\'quote\'][\'EUR\'][\'price\']),\n            \'provider_timestamp\': result.get(\'status\', {}).get(\'timestamp\'), \'fetched_at\': now()}\n\n\ndef run_chain(api, threshold, run_label, attempts):\n    if not re.fullmatch(r\'[a-z0-9]{1,32}\', run_label):\n        raise AccessError(\'Use a stable lowercase alphanumeric exercise run label.\')\n    threshold = finite_positive(threshold)\n    quote = fetch_btc(api)  # fresh request for each manual execution\n    identity = f\'bpa.btc.{run_label}@example.com\'\n    before = find_exact(api, identity)\n    if before:\n        raise AccessError(\'This chain run identity already exists; review it instead of creating again.\')\n    report = dict(quote, threshold_eur=threshold, condition=\'price > threshold\', run_label=run_label)\n    if above_threshold(quote[\'price_eur\'], threshold):\n        price = quote[\'price_eur\']\n        lead = {\'Last_Name\': f\'ÁRJELZÉS BTC {price:,.0f} EUR\',\n                \'Company\': \'Árfigyelő automatizmus\', \'Email\': identity, \'Lead_Source\': \'API\',\n                \'Description\': f\'Oktatási jelzés: BTC {price:.2f} EUR > {threshold:.2f} EUR.\'}\n        report.update(branch=\'positive\', crm_write_attempted=True,\n                      crm=create_verified(api, lead, attempts))\n    else:\n        after = find_exact(api, identity)\n        if after:\n            raise AccessError(\'Unexpected record appeared during negative check; reconcile CRM.\')\n        report.update(branch=\'negative\', crm_write_attempted=False,\n                      matching_records_before=0, matching_records_after=0)\n    return report\n', 'colab_runtime': '"""Memory-only adaptation of supplied token/refresh cells; no workspace secrets.\n\nAPI operations inherit the shared CourseAPI, including redirect rejection,\nheader-based CMC authentication and no automatic retry of CRM writes.\n"""\nfrom getpass import getpass\nimport json\nimport time\nfrom urllib.parse import urlencode\nfrom urllib.request import Request\nfrom urllib.error import HTTPError, URLError\n\nfrom bpa_access import AccessError, CourseAPI\n\n\nclass MemoryCourseAPI(CourseAPI):\n    def __init__(self, values):\n        super().__init__()\n        self._values = dict(values)\n        self._expires_at = 0\n\n    def credentials(self):\n        return self._values\n\n    def authorize(self, grant_code=\'\'):\n        values = self._values\n        fields = {\'client_id\': values.get(\'ZOHO_CLIENT_ID\', \'\'),\n                  \'client_secret\': values.get(\'ZOHO_CLIENT_SECRET\', \'\')}\n        if not all(fields.values()):\n            raise AccessError(\'Private runtime Client ID and Client Secret are required.\')\n        if grant_code:\n            fields.update(grant_type=\'authorization_code\', code=grant_code)\n        elif values.get(\'ZOHO_REFRESH_TOKEN\'):\n            fields.update(grant_type=\'refresh_token\', refresh_token=values[\'ZOHO_REFRESH_TOKEN\'])\n        else:\n            raise AccessError(\'Private runtime authorization is missing.\')\n        request = Request(\'https://accounts.zoho.eu/oauth/v2/token\',\n                          data=urlencode(fields).encode(\'utf-8\'), method=\'POST\',\n                          headers={\'Content-Type\': \'application/x-www-form-urlencoded\'})\n        try:\n            with self.opener.open(request, timeout=30) as response:\n                result = json.loads(response.read())\n        except (HTTPError, URLError, OSError, ValueError):\n            raise AccessError(\'EU token request failed; check authorization privately. No automatic code retry.\') from None\n        if not isinstance(result, dict) or result.get(\'api_domain\') != \'https://www.zohoapis.eu\' or not result.get(\'access_token\'):\n            raise AccessError(\'EU token response was not valid; check authorization privately.\')\n        try:\n            duration = int(result.get(\'expires_in\', 3600))\n        except (ValueError, TypeError):\n            raise AccessError(\'Token response has an invalid duration.\') from None\n        if duration <= 120:\n            raise AccessError(\'Token response has an unusable duration.\')\n        values[\'ZOHO_ACCESS_TOKEN\'] = result[\'access_token\']\n        if result.get(\'refresh_token\'):\n            values[\'ZOHO_REFRESH_TOKEN\'] = result[\'refresh_token\']\n        self._expires_at = time.time() + duration\n\n    def authenticate(self, *, force=False):\n        if force or not self._values.get(\'ZOHO_ACCESS_TOKEN\') or time.time() >= self._expires_at - 120:\n            self.authorize()\n        return self._values[\'ZOHO_ACCESS_TOKEN\']\n\n    def clear(self):\n        self._values.clear()\n        self._expires_at = 0\n\n\ndef private_runtime():\n    # Human fills masked prompts. Agent never transfers .env.local into Colab.\n    values = {\'ZOHO_CLIENT_ID\': getpass(\'Zoho EU Client ID: \').strip(),\n              \'ZOHO_CLIENT_SECRET\': getpass(\'Zoho EU Client Secret: \').strip()}\n    mode = input(\'Existing refresh token (r) or fresh unused grant code (g)? \').strip().lower()\n    grant = \'\'\n    if mode == \'r\':\n        values[\'ZOHO_REFRESH_TOKEN\'] = getpass(\'Existing Zoho refresh token: \').strip()\n    elif mode == \'g\':\n        grant = getpass(\'Fresh unused Zoho EU grant code: \').strip()\n        if not grant:\n            raise AccessError(\'Grant code cannot be empty.\')\n    else:\n        raise AccessError(\'Choose r or g; no request was sent.\')\n    api = MemoryCourseAPI(values)\n    values.clear()\n    try:\n        api.authorize(grant)\n    finally:\n        grant = \'\'\n    api.credentials()[\'CMC_PRO_API_KEY\'] = getpass(\'CoinMarketCap API key: \').strip()\n    if not api.credentials()[\'CMC_PRO_API_KEY\']:\n        api.clear()\n        raise AccessError(\'CMC key cannot be empty.\')\n    return api\n'}
for name, source in MODULE_SOURCES.items():
    module = types.ModuleType(name)
    module.__file__ = "/content/BPA/nap03/working/" + name + ".py"
    sys.modules[name] = module
    exec(compile(source, module.__file__, "exec"), module.__dict__)
from api_exercises import SAMPLE, MINI, create_verified, fetch_btc, run_chain, above_threshold
from colab_runtime import private_runtime
attempts = {}
results = {}
print("Közös kliens betöltve; még nem történt API-kérés.")


## Privát hozzáférés – a felhasználó tölti ki
A már működő EU-kliens újrahasználható. `r`: a meglévő refresh token; `g`: friss, egyszer használható grant code. A mezők értéke nem kerül kódba vagy kimenetbe. Az ügynök nem olvassa a bevitelt.


In [ ]:
api = private_runtime()
print("EU token kész a runtime memóriájában; titkos érték nincs kiírva.")


## API Teszt / Atlas Market
Egyszer futtasd. Meglévő pontos rekordnál csak ellenőrzés történik; az nem új létrehozás. Hálózati hiba után először ellenőrizd a CRM-et, ne indítsd újra a POST-ot.


In [ ]:
results["sample"] = create_verified(api, SAMPLE, attempts)
print(results["sample"])


## Mini-feladat – külön fiktív érdeklődő


In [ ]:
results["mini"] = create_verified(api, MINI, attempts)
print(results["mini"])


## Aktuális ár és kézzel választott küszöb
Az ág minden futása új árat kér. Válassz egy pozitív, az aktuális ár alatti küszöböt az első ághoz; a másodikhoz az aktuális ár felettit. Ha a piac közben megváltozik, a tényleges ág számít.


In [ ]:
quote = fetch_btc(api)
print(quote)


In [ ]:
positive_threshold = float(input("Pozitív ág küszöbe EUR (ponttal): "))
results["positive"] = run_chain(api, positive_threshold, "positive1", attempts)
print(results["positive"])


In [ ]:
negative_threshold = float(input("Negatív ág küszöbe EUR (ponttal): "))
results["negative"] = run_chain(api, negative_threshold, "negative1", attempts)
print(results["negative"])


## Külön offline határeset
Ez nem élő árfolyam- vagy CRM-bizonyíték.


In [ ]:
assert above_threshold(100, 99) is True
assert above_threshold(100, 100) is False
assert above_threshold(100, 101) is False
print("Offline szigorú > határesetek rendben.")


## Bizonyíték és lezárás
A CRM-ben nyisd meg a visszakapott rekordokat, ellenőrizd a mezőket és az API Teszt Timeline / Lead Created bejegyzését. A negatív ág kimenetében `crm_write_attempted=False` és mindkét azonosságvizsgálat 0 legyen. Az alábbi export csak a fiktív tanfolyami mezőket és eredményeket tartalmazza. A notebook kimenetét megosztás előtt töröld.


In [ ]:
import json
from google.colab import files
with open("BPA_Nap3_API_evidence.json", "w", encoding="utf-8") as stream:
    json.dump(results, stream, ensure_ascii=False, indent=2)
files.download("BPA_Nap3_API_evidence.json")


In [ ]:
api.clear()
print("Runtime API-kulcsok és tokenek törölve. A session bezárható.")
